> **Notebook file touched from disk: 2026-05-13 18:40 UTC** — if you do not see this line at the top, Cursor is showing a stale buffer: close tab → Command Palette → `File: Revert File` → reopen.

## Cursor: if Section 4 still looks like “one scatter only”

The notebook on disk **does** include **three chart code cells** (bar chart + two scatters) plus an optional combined figure. If you do not see them, the editor tab is usually **stale**.

**Try this (in order):**

1. **Close** this tab (do not keep an unsaved duplicate).
2. Command Palette → **File: Revert File** (loads from disk), **or** reopen `week6_mp1_starter.ipynb` from the Explorer.
3. If it still looks wrong, open **`MP1_WEEK6_CHARTS_COPY.ipynb`** in this same folder — it is a **byte-for-byte copy** and opens with a fresh buffer.
4. Press **Cmd+F** / **Ctrl+F** and search for **`barh`** — that jumps to **Chart 1** code.

---



# Mini Project 1 — Analysis Notebook

**Your name:** Swathi  
**Dataset:** Hardcover GraphQL books dataset  
**Date:** May 6, 2026  

This notebook uses the same Hardcover dataset and questions from my Week 5 A5 pandas analysis, then extends that work in MP1 format with clear interpretation and one supporting visualization.

In [ ]:
# Setup — run this cell first
# If any package is missing, it will install automatically
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["pandas", "plotly", "kaleido", "nbformat"]:
    try:
        __import__(pkg)
    except ImportError:
        print(f"Installing {pkg}...")
        install(pkg)

import pandas as pd
import plotly.express as px

print("Setup complete.")

---

## Section 1 — Overview

**Dataset:** Hardcover books data from the Hardcover GraphQL API ([https://api.hardcover.app/v1/graphql](https://api.hardcover.app/v1/graphql)). I query books, ratings distributions, and taggings (including genre tags).

**Why this dataset:** I am interested in how reader rating behavior differs across genres and how tagging patterns relate to perceived book quality.

**Three analytical questions:**

1. Which genre is associated with the most 1-star and 2-star ratings?
2. Which books appear most polarized versus widely distributed across rating buckets?
3. Is there a relationship between number of distinct tags and average rating?

**What a practitioner would do with these findings:** A product/data team at a reading platform could use this to prioritize recommendation logic, moderation, and discovery features based on disagreement patterns rather than only average ratings.

---

## Section 2 — Data Profile

Load your dataset and get a basic picture of what's in it. Answer these questions in a markdown cell below your code:

- How many rows and columns does your dataset have?
- What does each column represent?
- Are there any obvious data quality issues (missing values, unexpected types, inconsistent formatting)?
- Which column or columns will your analysis focus on, and why?

In [ ]:
# Load Hardcover books data from API (same approach as A5)
import json
import os
import ssl
import urllib.request
from pathlib import Path

import importlib.util
import subprocess
import sys

if importlib.util.find_spec("dotenv") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "python-dotenv"])

from dotenv import load_dotenv

# Look for token in local .env first, then week5 .env
env_candidates = [Path.cwd() / ".env", Path.cwd().parent / "hcde530_week5" / ".env"]
for env_path in env_candidates:
    if env_path.exists():
        load_dotenv(env_path, override=False)

TOKEN = os.environ.get("HARDCOVER_JWT")
if not TOKEN:
    raise SystemExit("Missing HARDCOVER_JWT in .env (week6 or week5)")

SSL_CTX = ssl.create_default_context()
try:
    SSL_CTX.load_verify_locations(cafile="/etc/ssl/cert.pem")
except OSError:
    pass

GRAPHQL_URL = "https://api.hardcover.app/v1/graphql"
LIMIT_PER_PAGE = 120
MAX_PAGES = 3


def fetch_books_page(limit: int, offset: int) -> list:
    query = """
    query Books($limit: Int!, $offset: Int!) {
      books(
        where: {
          ratings_count: { _gte: 40 }
          book_status_id: { _eq: 1 }
        }
        order_by: { ratings_count: desc }
        limit: $limit
        offset: $offset
      ) {
        id
        title
        rating
        ratings_count
        ratings_distribution
        taggings {
          tag {
            slug
            tag
            tag_category { slug category }
          }
        }
      }
    }
    """
    payload = {"query": query, "variables": {"limit": limit, "offset": offset}}
    req = urllib.request.Request(
        GRAPHQL_URL,
        data=bytes(json.dumps(payload), "utf-8"),
        headers={"Content-Type": "application/json", "Authorization": f"Bearer {TOKEN}"},
        method="POST",
    )
    with urllib.request.urlopen(req, context=SSL_CTX) as resp:
        body = json.loads(resp.read().decode())
    if "errors" in body:
        raise RuntimeError(body["errors"])
    return body["data"]["books"]


def book_rows_from_api(books: list) -> list:
    rows = []
    for b in books:
        dist = b.get("ratings_distribution") or []
        counts_by_star = {item["rating"]: item["count"] for item in dist}
        total = sum(counts_by_star.values())

        low_ct = counts_by_star.get(1.0, 0) + counts_by_star.get(2.0, 0)
        low_frac = low_ct / total if total else 0.0

        ent = 0.0
        if total:
            import math

            for c in counts_by_star.values():
                if c > 0:
                    p = c / total
                    ent -= p * math.log(p)

        mean = sum(r * c for r, c in counts_by_star.items()) / total if total else 0.0
        wvar = sum(c * (r - mean) ** 2 for r, c in counts_by_star.items()) / total if total else 0.0
        spread = wvar ** 0.5

        def _is_genre_category(cat: dict) -> bool:
            if not cat:
                return False
            slug_c = (cat.get("slug") or "").lower()
            name_c = (cat.get("category") or "").lower()
            return slug_c in ("genre", "genres") or name_c in ("genre", "genres")

        tag_slugs, genre_slugs = [], []
        for t in b.get("taggings") or []:
            tg = t.get("tag") or {}
            slug = tg.get("slug")
            if not slug:
                continue
            tag_slugs.append(slug)
            if _is_genre_category(tg.get("tag_category") or {}):
                genre_slugs.append(slug)

        uniq_tags = sorted(set(tag_slugs))
        uniq_genres = sorted(set(genre_slugs))

        n1, n5 = counts_by_star.get(1.0, 0), counts_by_star.get(5.0, 0)
        pct_1_or_5_star = (n1 + n5) / total if total else 0.0

        rows.append(
            {
                "id": b["id"],
                "title": b["title"],
                "avg_rating": float(b["rating"]) if b.get("rating") is not None else None,
                "ratings_count": b["ratings_count"],
                "low_star_ratings": int(low_ct),
                "low_star_fraction": round(low_frac, 5),
                "dist_entropy": round(ent, 5),
                "dist_spread_stdev": round(spread, 5),
                "pct_1_or_5_star": round(pct_1_or_5_star, 5),
                "polarized_1_or_5_ge_65pct": pct_1_or_5_star >= 0.65,
                "tag_slugs": "|".join(uniq_tags),
                "num_distinct_tags": len(uniq_tags),
                "genre_slugs": "|".join(uniq_genres),
                "num_distinct_genres": len(uniq_genres),
            }
        )
    return rows


books_raw = []
for page in range(MAX_PAGES):
    chunk = fetch_books_page(LIMIT_PER_PAGE, page * LIMIT_PER_PAGE)
    if not chunk:
        break
    books_raw.extend(chunk)

rows = book_rows_from_api(books_raw)
df = pd.DataFrame(rows).drop_duplicates(subset="id").reset_index(drop=True)

print("rows, cols:", df.shape)
df.head()

In [ ]:
# Check column types and missing values
print("Column types / non-null counts:")
df.info()

print("\nMissing values by column:")
df.isnull().sum().sort_values(ascending=False)

In [ ]:
# Summary statistics for numeric columns
df.describe(include="all").T.head(20)

**Your data profile notes:**  
The dataset contains one row per book with rating distribution metrics and tag metadata derived from Hardcover API results. The table includes core fields (`title`, `avg_rating`, `ratings_count`) plus engineered columns such as `low_star_ratings`, `dist_spread_stdev`, `pct_1_or_5_star`, and `num_distinct_tags`. Most key analysis columns are complete, which makes it practical to compare disagreement patterns across genres. A key caveat is that genre information is multi-valued (`genre_slugs` joined by `|`), so genre comparisons require split/explode logic rather than direct counting.

---

## Section 3 — Analysis

Answer your three research questions using pandas. Each question should have:

1. A markdown cell stating the question
2. A code cell with the analysis
3. A markdown cell with your interpretation — what does the result mean?

You may need to clean or reshape the data before you can answer a question. That's normal — document what you did and why.

**Question 1:** Which genre is associated with the most 1★ and 2★ ratings?

In [ ]:
# Split multi-genre strings to one row per book-genre pair, then aggregate low-star totals.
long = df.assign(genre=df["genre_slugs"].fillna("").str.split("|")).explode("genre")
long = long[long["genre"].notna() & (long["genre"] != "")]

q1_top = (
    long.groupby("genre", as_index=False)["low_star_ratings"]
    .sum()
    .sort_values("low_star_ratings", ascending=False)
)

q1_top.head(12)

**Interpretation:**  
In this sample, **fiction**, **fantasy**, and **young-adult** appear at the top for summed low-star counts. This suggests those genres carry the largest volume of 1★ and 2★ ratings, but that likely reflects both popularity and size of those genre buckets. A next step is to normalize by total ratings or books per genre so I can compare dissatisfaction rates more fairly, not just raw totals.

**Question 2:** Which books have the widest rating distributions, and which look most polarized toward 1★/5★?

In [ ]:
# Wide disagreement: rank by weighted spread of rating distribution.
q2_widest = df.sort_values("dist_spread_stdev", ascending=False)[
    [
        "title",
        "ratings_count",
        "avg_rating",
        "dist_spread_stdev",
        "dist_entropy",
        "pct_1_or_5_star",
        "polarized_1_or_5_ge_65pct",
    ]
].head(10)

print("Books with widest rating spread:")
q2_widest

# Polarization rule: >=65% of ratings in whole 1★ and 5★ buckets.
q2_polarized = df[df["polarized_1_or_5_ge_65pct"]].sort_values(
    "ratings_count", ascending=False
)[["title", "ratings_count", "avg_rating", "pct_1_or_5_star"]].head(10)

print("\nMostly 1★/5★ books (>=65% rule):")
q2_polarized

**Interpretation:**  
The spread and entropy rankings identify books where readers disagree strongly rather than simply giving low averages. Using a 65% threshold in the 1★+5★ buckets captures titles that look "love it or hate it" while still allowing middle-star long tails. This confirms that average rating alone misses important differences in how opinions are distributed.

**Question 3:** Is there a relationship between number of distinct tags and average rating?

In [ ]:
# Correlation between tagging richness and average rating.
q3_corr = df[["num_distinct_tags", "avg_rating"]].corr()
q3_corr

**Interpretation:**  
The correlation is small and positive in this sample, which suggests books with more distinct tags tend to have slightly higher average ratings. This is an association, not causation: richer tagging may reflect popularity, catalog coverage, or community behavior rather than quality itself. I would test this further by controlling for `ratings_count` and publication recency.

## Section 4 — Visualization *(A6: three charts + one optional combined cell)*

For **A6** you extend one MP1 figure into **multiple charts** tied to your MP1a questions. Below are **three runnable chart code cells** (plus one optional cell that draws all three at once). Each code cell includes its own imports so you can run it alone after `df` and `q1_top` exist.

Charts also save PNG files next to this notebook (`mp1_fig*.png`) for graders or editors that hide inline plots.

---


### Chart 1 — Low-star totals by genre (Question 1)

**Why a horizontal bar chart:** Genres are categorical; bars make it easy to compare summed 1★+2★ counts after exploding `genre_slugs`.

**Takeaway:** Big genres can dominate raw totals — compare magnitudes, not only rank.


In [ ]:
%matplotlib inline

import importlib.util
import subprocess
import sys
from pathlib import Path

if importlib.util.find_spec("matplotlib") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "matplotlib"])

import matplotlib.pyplot as plt


def _viz_dir() -> Path:
    """Folder that contains this notebook (cwd may be repo root or hcde530_week6/)."""
    here = Path.cwd()
    if (here / "week6_mp1_starter.ipynb").exists():
        return here
    if (here / "hcde530_week6" / "week6_mp1_starter.ipynb").exists():
        return here / "hcde530_week6"
    return here


_VIZ = _viz_dir()

top = q1_top.head(10).sort_values("low_star_ratings", ascending=True)
fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(top["genre"], top["low_star_ratings"], color="#4C72B0")
ax.set_title("Raw 1★+2★ counts are highest in big genres (exploded slugs)")
ax.set_xlabel("Summed 1★ + 2★ rating counts (attributed across genres)")
ax.set_ylabel("Genre slug")
plt.tight_layout()
fig.savefig(_VIZ / "mp1_fig1_lowstar_by_genre.png", dpi=150, bbox_inches="tight")
plt.show()


### Chart 2 — Rating spread vs average rating (Question 2)

**Why a scatter plot:** Both axes are numeric per book; scatter shows disagreement (`dist_spread_stdev`) separate from mean quality (`avg_rating`).

**Takeaway:** High spread can happen at many average ratings — polarization is not the same as “bad book.”


In [ ]:
%matplotlib inline

import importlib.util
import subprocess
import sys
from pathlib import Path

if importlib.util.find_spec("matplotlib") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "matplotlib"])

import matplotlib.pyplot as plt


def _viz_dir() -> Path:
    """Folder that contains this notebook (cwd may be repo root or hcde530_week6/)."""
    here = Path.cwd()
    if (here / "week6_mp1_starter.ipynb").exists():
        return here
    if (here / "hcde530_week6" / "week6_mp1_starter.ipynb").exists():
        return here / "hcde530_week6"
    return here


_VIZ = _viz_dir()

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(df["avg_rating"], df["dist_spread_stdev"], alpha=0.35, s=18, color="#C44E52")
ax.set_title("Wider star distributions (higher spread) vs average rating")
ax.set_xlabel("Average rating (Hardcover)")
ax.set_ylabel("Spread (weighted stdev across star buckets)")
plt.tight_layout()
fig.savefig(_VIZ / "mp1_fig2_spread_vs_avg_rating.png", dpi=150, bbox_inches="tight")
plt.show()


### Chart 3 — Distinct tags vs average rating (Question 3)

**Why a scatter plot:** Tests a simple association between tagging richness and mean rating.

**Takeaway:** Look for vertical scatter at each tag count — a weak trend still needs caution on causality.


In [ ]:
%matplotlib inline

import importlib.util
import subprocess
import sys
from pathlib import Path

if importlib.util.find_spec("matplotlib") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "matplotlib"])

import matplotlib.pyplot as plt


def _viz_dir() -> Path:
    """Folder that contains this notebook (cwd may be repo root or hcde530_week6/)."""
    here = Path.cwd()
    if (here / "week6_mp1_starter.ipynb").exists():
        return here
    if (here / "hcde530_week6" / "week6_mp1_starter.ipynb").exists():
        return here / "hcde530_week6"
    return here


_VIZ = _viz_dir()

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(df["num_distinct_tags"], df["avg_rating"], alpha=0.35, s=18, color="#55A868")
ax.set_title("Tag count vs average rating (association, not causation)")
ax.set_xlabel("Number of distinct tag slugs")
ax.set_ylabel("Average rating")
plt.tight_layout()
fig.savefig(_VIZ / "mp1_fig3_tags_vs_avg_rating.png", dpi=150, bbox_inches="tight")
plt.show()


### Optional — all three charts in one run

Runs the same three charts **one after another** — each in its **own** figure window and **own** PNG (`mp1_fig_all_three_1.png` … `_3.png`). Requires `df` and `q1_top` from Section 3.


In [ ]:
%matplotlib inline

import importlib.util
import subprocess
import sys
from pathlib import Path

if importlib.util.find_spec("matplotlib") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "matplotlib"])

import matplotlib.pyplot as plt


def _viz_dir() -> Path:
    """Folder that contains this notebook (cwd may be repo root or hcde530_week6/)."""
    here = Path.cwd()
    if (here / "week6_mp1_starter.ipynb").exists():
        return here
    if (here / "hcde530_week6" / "week6_mp1_starter.ipynb").exists():
        return here / "hcde530_week6"
    return here


_VIZ = _viz_dir()

top = q1_top.head(10).sort_values("low_star_ratings", ascending=True)

# Figure 1 — own window + own PNG (not a 1×3 strip)
fig1, ax = plt.subplots(figsize=(6, 5))
ax.barh(top["genre"], top["low_star_ratings"], color="#4C72B0")
ax.set_title("1★+2★ by genre")
ax.set_xlabel("Low-star sum")
fig1.tight_layout()
fig1.savefig(_VIZ / "mp1_fig_all_three_1.png", dpi=150, bbox_inches="tight")
plt.show()

fig2, ax = plt.subplots(figsize=(6, 4.5))
ax.scatter(df["avg_rating"], df["dist_spread_stdev"], alpha=0.35, s=14, color="#C44E52")
ax.set_title("Spread vs avg rating")
ax.set_xlabel("Avg rating")
ax.set_ylabel("Spread")
fig2.tight_layout()
fig2.savefig(_VIZ / "mp1_fig_all_three_2.png", dpi=150, bbox_inches="tight")
plt.show()

fig3, ax = plt.subplots(figsize=(6, 4.5))
ax.scatter(df["num_distinct_tags"], df["avg_rating"], alpha=0.35, s=14, color="#55A868")
ax.set_title("Tags vs avg rating")
ax.set_xlabel("Distinct tags")
ax.set_ylabel("Avg rating")
fig3.tight_layout()
fig3.savefig(_VIZ / "mp1_fig_all_three_3.png", dpi=150, bbox_inches="tight")
plt.show()


### Static PNG previews (if inline plots do not show)

Run the chart cells above first so these files exist in `hcde530_week6/`:

![Chart 1](mp1_fig1_lowstar_by_genre.png)

![Chart 2](mp1_fig2_spread_vs_avg_rating.png)

![Chart 3](mp1_fig3_tags_vs_avg_rating.png)

![All three — panel 1](mp1_fig_all_three_1.png)

![All three — panel 2](mp1_fig_all_three_2.png)

![All three — panel 3](mp1_fig_all_three_3.png)


**Chart rationales (A6):**  
Chart 1 uses **bars** for categorical genre totals. Charts 2 and 3 use **scatter** for numeric–numeric relationships. The optional cell runs all three charts in sequence (three separate images / PNGs). Together they mirror Questions 1–3 from Section 3.


---

## Section 5 — Conclusions

Write 3–5 sentences summarizing what you found. Address these questions:

- What is the most important thing your analysis revealed?
- What surprised you?
- What would you investigate next if you had more time or data?
- What are the limitations of this analysis — what can't you conclude from this data?

Then complete the competency claim below.

**Summary of findings:**  
This analysis shows that fiction, fantasy, and young-adult account for the largest raw volume of low-star ratings in the sampled Hardcover books. Distribution-based metrics reveal that some books are not just low-rated but highly polarized, with a large share of 1★ and 5★ ratings. I also found a weak positive association between number of distinct tags and average rating, but with wide scatter and clear limits on causal interpretation. The most important next step is normalization (for genre size and rating volume) and additional controls before making product decisions from these patterns.

---

## Competency Claim

In a `mp1.md` file in your GitHub repository, write a short competency claim (2–4 sentences) for each domain you feel this project demonstrates. Be specific — cite something you actually did in this notebook.

Domains covered by this project typically include:
- **C3 — Data cleaning and file handling** (if you cleaned or reshaped data)
- **C5 — Data analysis with pandas** (answering questions with code)
- **C6 — Data visualization** (your chart)
- **C7 — Critical evaluation and professional judgment** (your interpretation and limitations section)

You don't have to claim every domain — only the ones your work actually demonstrates.